# CoCIS Real-Time Customer Care Service Voice Assistant
## Model Training Notebook Whisper (ASR) & wav2vec 2.0 (Language ID)
**Makerere University | College of Computing and Information Sciences**  
**Dataset:** Sunbird AI SALT (sunbird/salt) - 24,912 clean samples | 7 Ugandan Languages  
**Author:** AKOL PAUL 22/U/22453 | **Date:** March 2026

Notebook Structure
| Section | Model | Task |
|---------|-------|------|
| **PART A** | wav2vec 2.0 (XLSR-53) | Language Identification (7-class) |
| **PART B** | Whisper (small) | Automatic Speech Recognition (ASR) |


# PART A wav2vec 2.0 | Language Identification


In [ ]:
import os

W2V_OUTPUT_DIR     = '/content/wav2vec2_lang_id'
WHISPER_OUTPUT_DIR = '/content/whisper_asr'
W2V_CACHE_DIR      = '/content/salt_cache_w2v'
WHISPER_CACHE_DIR  = '/content/salt_cache_whisper'

for d in [W2V_OUTPUT_DIR, WHISPER_OUTPUT_DIR, W2V_CACHE_DIR, WHISPER_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(' Output dirs ready — saving to /content/')
print()
print(' IMPORTANT: Download your models manually after training finishes.')
print('   Use this code after each model saves:')
print()
print('   from google.colab import files')
print('   files.download("/content/wav2vec2_lang_id/final_model/model.safetensors")')
print('   files.download("/content/whisper_asr/final_model/model.safetensors")')

 Output dirs ready — saving to /content/

 IMPORTANT: Download your models manually after training finishes.
   Use this code after each model saves:

   from google.colab import files
   files.download("/content/wav2vec2_lang_id/final_model/model.safetensors")
   files.download("/content/whisper_asr/final_model/model.safetensors")


In [ ]:
# CELL A-1 │ Install Dependencies
!pip install transformers datasets huggingface_hub evaluate accelerate librosa soundfile jiwer -q
print('All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.4 MB/s eta 0:00:00
All packages installed


In [ ]:
# CELL A-2
# Imports
import os, gc, pickle, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset as TorchDataset
from datasets import load_dataset, Audio
from huggingface_hub import login
from google.colab import userdata
import evaluate
import librosa
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f' Imports complete | Device: {DEVICE}')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None enable GPU in Settings"}')

 Imports complete | Device: cuda
   GPU: Tesla T4


In [ ]:
# CELL A-3 │ HuggingFace Authentication
HF_TOKEN = userdata.get('appauli_sunbird_access_token')
print(f'Token found: {HF_TOKEN is not None}')
login(token=HF_TOKEN)
print(' Logged in to HuggingFace')

Token found: True
 Logged in to HuggingFace


In [ ]:
# CELL A-4 │ Configuration wav2vec 2.0
LANGUAGES = ['studio-ach', 'studio-swa', 'studio-lug',
             'studio-lgg', 'studio-teo', 'studio-eng', 'studio-nyn']

LABEL_MAP = {
    'studio-ach': 0, 'studio-swa': 1, 'studio-lug': 2,
    'studio-lgg': 3, 'studio-teo': 4, 'studio-eng': 5, 'studio-nyn': 6,
}
ID2LABEL   = {v: k.replace('studio-', '') for k, v in LABEL_MAP.items()}
LABEL2ID   = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(LABEL_MAP)

# Audio
SAMPLE_RATE  = 16_000
MAX_DURATION = 20.0
MAX_LENGTH   = int(SAMPLE_RATE * MAX_DURATION)
MIN_DURATION = 0.5

# Model
W2V_BASE_MODEL = 'facebook/wav2vec2-large-xlsr-53'
W2V_OUTPUT_DIR = '/content/wav2vec2_lang_id'
W2V_CACHE_DIR  = '/content/salt_cache_w2v'
os.makedirs(W2V_OUTPUT_DIR, exist_ok=True)
os.makedirs(W2V_CACHE_DIR,  exist_ok=True)

# Hyperparameters
W2V_LEARNING_RATE     = 3e-5
W2V_ENCODER_LR        = 1e-5
W2V_HEAD_LR           = 1e-4
W2V_BATCH_SIZE        = 8
W2V_GRAD_ACCUM        = 4
W2V_EPOCHS            = 15
W2V_WARMUP_STEPS      = 500
W2V_WEIGHT_DECAY      = 0.01
W2V_DROPOUT           = 0.1
W2V_ATTENTION_DROPOUT = 0.05
W2V_GRAD_CLIP         = 1.0
W2V_FREEZE_EPOCHS     = 3

print(' wav2vec 2.0 configuration set')
print(f'   Languages        : {NUM_LABELS} - {list(ID2LABEL.values())}')
print(f'   Base model       : {W2V_BASE_MODEL}')
print(f'   Effective batch  : {W2V_BATCH_SIZE * W2V_GRAD_ACCUM}')

 wav2vec 2.0 configuration set
   Languages        : 7 - ['ach', 'swa', 'lug', 'lgg', 'teo', 'eng', 'nyn']
   Base model       : facebook/wav2vec2-large-xlsr-53
   Effective batch  : 32


In [ ]:
# CELL A-5 │ Cache SALT to Disk - One Language at a Time
# Loads one language, saves waveforms to .pkl, frees RAM

def process_and_cache_lang_w2v(lang):
    save_path = f'{W2V_CACHE_DIR}/{lang}.pkl'
    if os.path.exists(save_path):
        print(f'  {lang}: already cached')
        return

    print(f'  Loading {lang}...', end=' ')
    ds    = load_dataset('sunbird/salt', lang)
    label = LABEL_MAP[lang]
    result = {'train': [], 'dev': [], 'test': []}

    for split in ['train', 'dev', 'test']:
        ds_split = ds[split].cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
        for sample in ds_split:
            waveform = sample['audio']['array'].astype(np.float32)

            # Pad very short clips
            if len(waveform) < int(MIN_DURATION * SAMPLE_RATE):
                waveform = np.pad(waveform,
                    (0, int(MIN_DURATION * SAMPLE_RATE) - len(waveform)))

            # Trim silence
            waveform, _ = librosa.effects.trim(waveform, top_db=40)

            # Truncate long clips (handles Lugbara avg 17.5s)
            if len(waveform) > MAX_LENGTH:
                waveform = waveform[:MAX_LENGTH]

            result[split].append({'waveform': waveform, 'labels': label})

        print(f'{split}={len(result[split])}', end=' ')

    with open(save_path, 'wb') as f:
        pickle.dump(result, f)

    del ds, result
    gc.collect()
    print('saved')


print('Caching SALT dataset language by language...')
for lang in LANGUAGES:
    process_and_cache_lang_w2v(lang)

print(f'\nAll languages cached to {W2V_CACHE_DIR}')
for fname in sorted(os.listdir(W2V_CACHE_DIR)):
    size = os.path.getsize(f'{W2V_CACHE_DIR}/{fname}') / (1024**2)
    print(f'   {fname:<30} {size:.1f} MB')

Caching SALT dataset language by language...
  Loading studio-ach... 

README.md: 0.00B [00:00, ?B/s]

studio-ach/train-00000-of-00001.parquet:   0%|          | 0.00/199M [00:00<?, ?B/s]

studio-ach/dev-00000-of-00001.parquet:   0%|          | 0.00/4.16M [00:00<?, ?B/s]

studio-ach/test-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

...

studio-nyn/test-00000-of-00001.parquet:   0%|          | 0.00/3.54M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3326 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/72 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/62 [00:00<?, ? examples/s]

train=3326 dev=72 test=62 saved

All languages cached to /content/salt_cache_w2v
   studio-ach.pkl                 987.6 MB
   studio-eng.pkl                 781.2 MB
   studio-lgg.pkl                 619.1 MB
   studio-lug.pkl                 573.8 MB
   studio-nyn.pkl                 1142.5 MB
   studio-swa.pkl                 998.0 MB
   studio-teo.pkl                 1230.5 MB


In [ ]:
# CELL A-5b │ PyTorch Dataset from Disk Cache wav2vec 2.0
# Load feature extractor first (needed inside dataset)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    W2V_BASE_MODEL, return_attention_mask=True
)

class SALTDiskDatasetW2V(TorchDataset):
    """
    Reads pre-cached waveforms from disk one language at a time.
    Runs Wav2Vec2FeatureExtractor on each waveform on-the-fly in __getitem__
    so no large preprocessed tensors sit in RAM.
    """
    def __init__(self, split, cache_dir=W2V_CACHE_DIR, languages=LANGUAGES):
        self.samples = []
        for lang in languages:
            with open(f'{cache_dir}/{lang}.pkl', 'rb') as f:
                data = pickle.load(f)
            self.samples.extend(data[split])
            del data
            gc.collect()

        if split == 'train':
            random.seed(42)
            random.shuffle(self.samples)

        print(f'  {split}: {len(self.samples):,} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item     = self.samples[idx]
        waveform = item['waveform']
        label    = item['labels']

        inputs = feature_extractor(
            waveform,
            sampling_rate=SAMPLE_RATE,
            return_tensors='pt',
            padding=False,
        )
        return {
            'input_values': inputs.input_values.squeeze(0),
            'labels':       torch.tensor(label, dtype=torch.long)
        }


print('Building datasets from disk cache...')
train_ds_proc = SALTDiskDatasetW2V('train')
dev_ds_proc   = SALTDiskDatasetW2V('dev')
test_ds_proc  = SALTDiskDatasetW2V('test')
print(f'\n Datasets ready')

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Building datasets from disk cache...
  train: 23,969 samples
  dev: 505 samples
  test: 459 samples

 Datasets ready


In [ ]:
# CELL A-6 │ Class Weights Lugbara Imbalance Correction
labels_array  = np.array([s['labels'] for s in train_ds_proc.samples])
N_total       = len(labels_array)
class_weights = []

print(f'{"Label":<5} {"Language":<12} {"N samples":>10} {"Weight":>8}')
print('-' * 40)
for i in range(NUM_LABELS):
    n_i    = np.sum(labels_array == i)
    weight = N_total / (NUM_LABELS * n_i)
    class_weights.append(weight)
    print(f'  {i:<3} {ID2LABEL[i]:<12} {n_i:>10,} {weight:>8.3f}')

CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f'\n Class weights computed')

Label Language      N samples   Weight
----------------------------------------
  0   ach               4,803    0.713
  1   swa               3,835    0.893
  2   lug               2,395    1.430
  3   lgg               1,594    2.148
  4   teo               4,649    0.737
  5   eng               3,367    1.017
  6   nyn               3,326    1.030

 Class weights computed


In [ ]:
# CELL A-7 │ Data Collator — Variable-Length Padding
# Pads all input_values in a batch to the same length
@dataclass
class Wav2Vec2CollatorWithPadding:
    feature_extractor: Any
    padding: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [
            {'input_values': f['input_values']} for f in features
        ]
        labels = torch.tensor(
            [f['labels'] for f in features], dtype=torch.long
        )
        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors='pt',
        )
        batch['labels'] = labels
        return batch


w2v_data_collator = Wav2Vec2CollatorWithPadding(feature_extractor=feature_extractor)
print('Wav2Vec2CollatorWithPadding ready')

Wav2Vec2CollatorWithPadding ready


In [ ]:
#  CELL A-8 │ Load Pretrained Model - wav2vec 2.0 XLSR-53
gc.collect()
torch.cuda.empty_cache()

w2v_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    W2V_BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    hidden_dropout=W2V_DROPOUT,
    attention_dropout=W2V_ATTENTION_DROPOUT,
    ignore_mismatched_sizes=True
)

# Phase 1: freeze CNN feature encoder
w2v_model.freeze_feature_encoder()
w2v_model = w2v_model.to(DEVICE)

total_params     = sum(p.numel() for p in w2v_model.parameters())
trainable_params = sum(p.numel() for p in w2v_model.parameters() if p.requires_grad)
print(f'   wav2vec 2.0 loaded | Encoder FROZEN for first {W2V_FREEZE_EPOCHS} epochs')
print(f'   Total parameters   : {total_params:,}')
print(f'   Trainable (phase 1): {trainable_params:,}')

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   wav2vec 2.0 loaded | Encoder FROZEN for first 3 epochs
   Total parameters   : 315,702,919
   Trainable (phase 1): 311,492,743


In [ ]:
# CELL A-9 │ WeightedTrainer + Metrics - wav2vec 2.0
metric_f1  = evaluate.load('f1')
metric_acc = evaluate.load('accuracy')

def compute_metrics_w2v(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1    = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    acc   = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    per_lang_f1 = metric_f1.compute(predictions=preds, references=labels, average=None)['f1']
    result = {'macro_f1': f1, 'accuracy': acc}
    for i, lf1 in enumerate(per_lang_f1):
        result[f'f1_{ID2LABEL[i]}'] = round(lf1, 4)
    return result


class WeightedTrainer(Trainer):
    """Injects class weights into CrossEntropyLoss to handle Lugbara imbalance."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


print('WeightedTrainer and compute_metrics defined')

WeightedTrainer and compute_metrics defined


In [ ]:
# CELL A-10 │ PHASE 1 Training - Encoder Frozen  (Epochs 1-3)
gc.collect()
torch.cuda.empty_cache()

phase1_args = TrainingArguments(
    output_dir=W2V_OUTPUT_DIR + '/phase1',
    num_train_epochs=W2V_FREEZE_EPOCHS,
    per_device_train_batch_size=W2V_BATCH_SIZE,
    per_device_eval_batch_size=W2V_BATCH_SIZE,
    gradient_accumulation_steps=W2V_GRAD_ACCUM,
    learning_rate=W2V_LEARNING_RATE,
    warmup_steps=W2V_WARMUP_STEPS,
    weight_decay=W2V_WEIGHT_DECAY,
    max_grad_norm=W2V_GRAD_CLIP,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to='none',
    seed=42
)

phase1_trainer = WeightedTrainer(
    model=w2v_model,
    args=phase1_args,
    train_dataset=train_ds_proc,
    eval_dataset=dev_ds_proc,
    compute_metrics=compute_metrics_w2v,
    data_collator=w2v_data_collator,
)

print('Starting Phase 1 training (encoder FROZEN)...')
phase1_result = phase1_trainer.train()
print(f'\n Phase 1 complete')
print(f'   Loss    : {phase1_result.training_loss:.4f}')
print(f'   Runtime : {phase1_result.metrics["train_runtime"]:.1f}s')

NameError: name 'gc' is not defined

In [ ]:
import gc, torch

# Delete Phase 1 trainer and results to free VRAM
del phase1_trainer
del phase1_result

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

allocated = torch.cuda.memory_allocated() / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
free      = total - allocated

print(f'GPU memory freed')
print(f'   Allocated : {allocated:.2f} GB')
print(f'   Free      : {free:.2f} GB')
print(f'   Status    : {"OK to proceed" if free > 4.0 else "Still low — restart runtime"}')

NameError: name 'phase1_trainer' is not defined

In [ ]:
import torch, gc, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

gc.collect()
torch.cuda.empty_cache()

allocated = torch.cuda.memory_allocated() / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
free      = total - allocated

print(f'Allocated : {allocated:.2f} GB')
print(f'Free      : {free:.2f} GB')
print(f'Status    : {"OK to train" if free > 4.0 else "Low — proceed carefully"}')

Allocated : 0.00 GB
Free      : 14.56 GB
Status    : OK to train


In [ ]:
# CELL A-11 │ PHASE 2 Training Full Model Unfrozen  (Epochs 4-15)
from torch.optim import AdamW as TorchAdamW
from transformers import get_linear_schedule_with_warmup

# Unfreeze encoder
for param in w2v_model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = True
print(f'Encoder UNFROZEN | Trainable: {sum(p.numel() for p in w2v_model.parameters() if p.requires_grad):,}')

# Differential learning rates
optimizer = TorchAdamW([
    {'params': w2v_model.wav2vec2.feature_extractor.parameters(), 'lr': W2V_ENCODER_LR},
    {'params': w2v_model.wav2vec2.encoder.parameters(),            'lr': W2V_LEARNING_RATE},
    {'params': w2v_model.classifier.parameters(),                  'lr': W2V_HEAD_LR},
], weight_decay=W2V_WEIGHT_DECAY)

total_steps = (len(train_ds_proc) // W2V_BATCH_SIZE) * (W2V_EPOCHS - W2V_FREEZE_EPOCHS)
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=W2V_WARMUP_STEPS, num_training_steps=total_steps
)

phase2_args = TrainingArguments(
    output_dir=W2V_OUTPUT_DIR + '/phase2',
    num_train_epochs=W2V_EPOCHS - W2V_FREEZE_EPOCHS,
    per_device_train_batch_size=W2V_BATCH_SIZE,
    per_device_eval_batch_size=W2V_BATCH_SIZE,
    gradient_accumulation_steps=W2V_GRAD_ACCUM,
    weight_decay=W2V_WEIGHT_DECAY,
    max_grad_norm=W2V_GRAD_CLIP,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to='none',
    seed=42
)

phase2_trainer = WeightedTrainer(
    model=w2v_model,
    args=phase2_args,
    train_dataset=train_ds_proc,
    eval_dataset=dev_ds_proc,
    compute_metrics=compute_metrics_w2v,
    data_collator=w2v_data_collator,
    optimizers=(optimizer, scheduler)
)

print('\nStarting Phase 2 training (full model UNFROZEN)...')
phase2_result = phase2_trainer.train()
print(f'\nnPhase 2 complete')
print(f'   Loss    : {phase2_result.training_loss:.4f}')
print(f'   Runtime : {phase2_result.metrics["train_runtime"]:.1f}s')

Encoder UNFROZEN | Trainable: 315,702,919

Starting Phase 2 training (full model UNFROZEN)...


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.12 GiB is allocated by PyTorch, and 301.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# CELL A-12 │ Evaluate on Test Set + Fairness Audit
print('Evaluating on held-out test set...')
test_results = phase2_trainer.evaluate(test_ds_proc)

print('\n' + '═' * 55)
print(' wav2vec 2.0 Language Identification — Test Results ')
print('═' * 55)
print(f'  Accuracy  : {test_results.get("eval_accuracy", 0):.4f}')
print(f'  Macro F1  : {test_results.get("eval_macro_f1", 0):.4f}')
print()

f1_scores = [test_results.get(f'eval_f1_{ID2LABEL[i]}', 0) for i in range(NUM_LABELS)]
best_f1   = max(f1_scores)
print(' Per-Language F1 (Fairness Audit — threshold: 10pp):')
print(f'  {"Language":<12} {"F1":>8}  Status')
print(f'  {"-"*35}')
for i, f1 in enumerate(f1_scores):
    status = 'PASS' if best_f1 - f1 <= 0.10 else '  FAIL'
    print(f'  {ID2LABEL[i]:<12} {f1:>8.4f}  {status}')

gap = best_f1 - min(f1_scores)
print(f'\n  Max F1 gap : {gap:.4f} | Threshold: 0.10')
print('  FAIRNESS  :', ' All within threshold' if gap <= 0.10 else 'Augmentation needed')

In [ ]:
#  CELL A-13 │ Save Model wav2vec 2.0
save_path = W2V_OUTPUT_DIR + '/final_model'
w2v_model.save_pretrained(save_path)
feature_extractor.save_pretrained(save_path)
print(f'wav2vec 2.0 saved → {save_path}')
for f in os.listdir(save_path):
    print(f'   {f}')

PART B Whisper | Automatic Speech Recognition


In [ ]:
# CELL B-2 │ Imports - Whisper
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate as hf_evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f' Whisper imports complete | Device: {DEVICE}')

In [ ]:
# CELL B-3 │ Configuration - Whisper
WHISPER_LANGUAGES = ['studio-ach', 'studio-swa', 'studio-lug',
                     'studio-lgg', 'studio-teo', 'studio-eng', 'studio-nyn']

LANG_TOKEN_MAP = {
    'studio-ach': 'ach', 'studio-swa': 'sw',  'studio-lug': 'lg',
    'studio-lgg': 'lgg', 'studio-teo': 'teo', 'studio-eng': 'en',
    'studio-nyn': 'nyn',
}

WHISPER_SAMPLE_RATE    = 16_000
WHISPER_MAX_DURATION   = 30.0
WHISPER_CHUNK_OVERLAP  = 2.0

WHISPER_MODEL      = 'openai/whisper-small'
WHISPER_OUTPUT_DIR = '/content/whisper_asr'
WHISPER_CACHE_DIR  = '/content/salt_cache_whisper'
os.makedirs(WHISPER_OUTPUT_DIR, exist_ok=True)
os.makedirs(WHISPER_CACHE_DIR,  exist_ok=True)

WHISPER_LR              = 1e-5
WHISPER_WARMUP_STEPS    = 500
WHISPER_BATCH_SIZE      = 8
WHISPER_GRAD_ACCUM      = 4       # effective batch = 32
WHISPER_EPOCHS          = 10
WHISPER_LABEL_SMOOTHING = 0.1
WHISPER_WEIGHT_DECAY    = 0.01
WHISPER_GRAD_CLIP       = 1.0
WHISPER_FREEZE_EPOCHS   = 2
WER_THRESHOLD           = 30.0

print('  Whisper configuration set')
print(f'   Base model      : {WHISPER_MODEL}')
print(f'   Effective batch : {WHISPER_BATCH_SIZE * WHISPER_GRAD_ACCUM}')

In [ ]:
#  CELL B-4 │ Cache SALT to Disk - One Language at a Time (Whisper)

def chunk_audio(waveform, sample_rate, max_dur=30.0, overlap=2.0):
    """Splits audio longer than max_dur into overlapping chunks."""
    max_samples = int(max_dur * sample_rate)
    overlap_samples = int(overlap * sample_rate)
    stride = max_samples - overlap_samples
    if len(waveform) <= max_samples:
        return [waveform]
    chunks, start = [], 0
    while start < len(waveform):
        end = min(start + max_samples, len(waveform))
        chunks.append(waveform[start:end])
        if end == len(waveform):
            break
        start += stride
    return chunks

def normalise_text(text):
    return re.sub(r'\s+', ' ', text.lower().strip())

def process_and_cache_lang_whisper(lang):
    save_path = f'{WHISPER_CACHE_DIR}/{lang}.pkl'
    if os.path.exists(save_path):
        print(f'  {lang}: already cached ✓')
        return

    print(f'  Loading {lang}...', end=' ')
    ds         = load_dataset('sunbird/salt', lang)
    lang_token = LANG_TOKEN_MAP[lang]
    result     = {'train': [], 'dev': [], 'test': []}

    for split in ['train', 'dev', 'test']:
        ds_split = ds[split].cast_column('audio', Audio(sampling_rate=WHISPER_SAMPLE_RATE))
        for sample in ds_split:
            waveform   = sample['audio']['array'].astype(np.float32)
            transcript = normalise_text(sample['text'])
            # Use first chunk for training (handles Lugbara >30s clips)
            chunks   = chunk_audio(waveform, WHISPER_SAMPLE_RATE,
                                   max_dur=WHISPER_MAX_DURATION,
                                   overlap=WHISPER_CHUNK_OVERLAP)
            waveform = chunks[0]
            result[split].append({
                'waveform':   waveform,
                'transcript': transcript,
                'language':   lang_token
            })
        print(f'{split}={len(result[split])}', end=' ')

    with open(save_path, 'wb') as f:
        pickle.dump(result, f)
    del ds, result
    gc.collect()
    print('saved')


print('Caching SALT for Whisper language by language...')
for lang in WHISPER_LANGUAGES:
    process_and_cache_lang_whisper(lang)

print(f'\n All languages cached to {WHISPER_CACHE_DIR}')
for fname in sorted(os.listdir(WHISPER_CACHE_DIR)):
    size = os.path.getsize(f'{WHISPER_CACHE_DIR}/{fname}') / (1024**2)
    print(f'   {fname:<30} {size:.1f} MB')

In [ ]:
# CELL B-5 │ Processor & Dataset from Disk Cache - Whisper
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL, language=None, task='transcribe')

class SALTDiskDatasetWhisper(TorchDataset):
    """
    Reads pre-cached waveforms from disk and processes on-the-fly:
    waveform → log-Mel spectrogram features + tokenised transcript labels.
    """
    def __init__(self, split, cache_dir=WHISPER_CACHE_DIR, languages=WHISPER_LANGUAGES):
        self.samples = []
        for lang in languages:
            with open(f'{cache_dir}/{lang}.pkl', 'rb') as f:
                data = pickle.load(f)
            self.samples.extend(data[split])
            del data
            gc.collect()

        if split == 'train':
            random.seed(42)
            random.shuffle(self.samples)

        print(f'  {split}: {len(self.samples):,} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item       = self.samples[idx]
        waveform   = item['waveform']
        transcript = item['transcript']
        language   = item['language']

        # Log-Mel spectrogram (80 mels, 25ms window)
        features = processor.feature_extractor(
            waveform, sampling_rate=WHISPER_SAMPLE_RATE, return_tensors='np'
        ).input_features[0]   # (80, 3000)

        # Tokenise transcript with language token
        processor.tokenizer.set_prefix_tokens(language=language, task='transcribe')
        token_ids = processor.tokenizer(
            transcript, return_tensors='np'
        ).input_ids[0]

        return {
            'input_features': torch.tensor(features, dtype=torch.float32),
            'labels':         torch.tensor(token_ids, dtype=torch.long),
            'language':       language
        }


print('Building Whisper datasets from disk cache...')
w_train = SALTDiskDatasetWhisper('train')
w_dev   = SALTDiskDatasetWhisper('dev')
w_test  = SALTDiskDatasetWhisper('test')
print(f'\n Whisper datasets ready')

In [ ]:
# CELL B-6 │ Data Collator - Whisper
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Pad input features
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        # Pad label sequences
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        # Replace padding token with -100 so it is ignored in loss
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove BOS token if prepended
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch


whisper_collator = WhisperDataCollator(processor=processor)
print(' WhisperDataCollator ready')

In [ ]:
# CELL B-7 │ Load Pretrained Whisper Small
gc.collect()
torch.cuda.empty_cache()

whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)
whisper_model.config.forced_decoder_ids = None
whisper_model.config.suppress_tokens    = []
whisper_model.config.use_cache          = False

# Phase 1: freeze encoder
for param in whisper_model.model.encoder.parameters():
    param.requires_grad = False

whisper_model = whisper_model.to(DEVICE)

total     = sum(p.numel() for p in whisper_model.parameters())
trainable = sum(p.numel() for p in whisper_model.parameters() if p.requires_grad)
print(f' Whisper loaded | Encoder FROZEN for first {WHISPER_FREEZE_EPOCHS} epochs')
print(f'   Total parameters   : {total:,}')
print(f'   Trainable (phase 1): {trainable:,}')

In [ ]:
#  CELL B-8 │ WER Metric - Whisper
wer_metric = hf_evaluate.load('wer')

def compute_metrics_whisper(eval_pred):
    pred_ids, label_ids = eval_pred
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str  = [normalise_text(s) for s in pred_str]
    label_str = [normalise_text(s) for s in label_str]
    pairs = [(p, r) for p, r in zip(pred_str, label_str) if r.strip()]
    if not pairs:
        return {'wer': 1.0}
    preds, refs = zip(*pairs)
    wer = wer_metric.compute(predictions=list(preds), references=list(refs))
    return {'wer': round(wer, 4)}

print('WER metric defined')

In [ ]:
# CELL B-9 │ PHASE 1 Training - Encoder Frozen  (Epochs 1-2)
gc.collect()
torch.cuda.empty_cache()

phase1_whisper_args = Seq2SeqTrainingArguments(
    output_dir=WHISPER_OUTPUT_DIR + '/phase1',
    num_train_epochs=WHISPER_FREEZE_EPOCHS,
    per_device_train_batch_size=WHISPER_BATCH_SIZE,
    per_device_eval_batch_size=WHISPER_BATCH_SIZE,
    gradient_accumulation_steps=WHISPER_GRAD_ACCUM,
    learning_rate=WHISPER_LR,
    warmup_steps=WHISPER_WARMUP_STEPS,
    weight_decay=WHISPER_WEIGHT_DECAY,
    max_grad_norm=WHISPER_GRAD_CLIP,
    label_smoothing_factor=WHISPER_LABEL_SMOOTHING,
    predict_with_generate=True,
    generation_max_length=225,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to='none',
    seed=42
)

phase1_whisper_trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=phase1_whisper_args,
    train_dataset=w_train,
    eval_dataset=w_dev,
    data_collator=whisper_collator,
    compute_metrics=compute_metrics_whisper,
    tokenizer=processor.feature_extractor
)

print('Starting Whisper Phase 1 training (encoder FROZEN)...')
p1_result = phase1_whisper_trainer.train()
print(f'\n Whisper Phase 1 complete')
print(f'   Loss    : {p1_result.training_loss:.4f}')
print(f'   Runtime : {p1_result.metrics["train_runtime"]:.1f}s')

In [ ]:
# CELL B-10 │ PHASE 2 Training  Full Model Unfrozen  (Epochs 3–10)
for param in whisper_model.model.encoder.parameters():
    param.requires_grad = True
print(f'Encoder UNFROZEN | Trainable: {sum(p.numel() for p in whisper_model.parameters() if p.requires_grad):,}')

phase2_whisper_args = Seq2SeqTrainingArguments(
    output_dir=WHISPER_OUTPUT_DIR + '/phase2',
    num_train_epochs=WHISPER_EPOCHS - WHISPER_FREEZE_EPOCHS,
    per_device_train_batch_size=WHISPER_BATCH_SIZE,
    per_device_eval_batch_size=WHISPER_BATCH_SIZE,
    gradient_accumulation_steps=WHISPER_GRAD_ACCUM,
    learning_rate=WHISPER_LR,
    warmup_steps=WHISPER_WARMUP_STEPS,
    weight_decay=WHISPER_WEIGHT_DECAY,
    max_grad_norm=WHISPER_GRAD_CLIP,
    label_smoothing_factor=WHISPER_LABEL_SMOOTHING,
    predict_with_generate=True,
    generation_max_length=225,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to='none',
    seed=42
)

phase2_whisper_trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=phase2_whisper_args,
    train_dataset=w_train,
    eval_dataset=w_dev,
    data_collator=whisper_collator,
    compute_metrics=compute_metrics_whisper,
    tokenizer=processor.feature_extractor
)

print('\nStarting Whisper Phase 2 training (full model UNFROZEN)...')
p2_result = phase2_whisper_trainer.train()
print(f'\n Whisper Phase 2 complete')
print(f'   Loss    : {p2_result.training_loss:.4f}')
print(f'   Runtime : {p2_result.metrics["train_runtime"]:.1f}s')

In [ ]:
#  CELL B-11 │ Evaluate on Test Set + WER Fairness Audit
print('Evaluating Whisper on held-out test set...')
overall = phase2_whisper_trainer.evaluate(w_test)
overall_wer = overall.get('eval_wer', None)

print('\n' + '═' * 55)
print(' Whisper ASR — Test Results                          ')
print('═' * 55)
print(f'  Overall WER: {overall_wer * 100:.2f}%' if overall_wer else '  Overall WER: N/A')
print()

# Per-language WER
lang_wers = {}
print(' Per-Language WER (Fairness Audit — threshold: 15pp):')
print(f'  {"Language":<12} {"WER":>8}  Status')
print(f'  {"-"*40}')

for lang_cfg, lang_code in LANG_TOKEN_MAP.items():
    lang_test = [s for s in w_test.samples if s['language'] == lang_code]
    if not lang_test:
        continue

    # Build a small temp dataset for this language
    class SingleLangDataset(TorchDataset):
        def __init__(self, samples):
            self.samples = samples
        def __len__(self):
            return len(self.samples)
        def __getitem__(self, idx):
            item = self.samples[idx]
            features = processor.feature_extractor(
                item['waveform'], sampling_rate=WHISPER_SAMPLE_RATE, return_tensors='np'
            ).input_features[0]
            processor.tokenizer.set_prefix_tokens(language=item['language'], task='transcribe')
            token_ids = processor.tokenizer(item['transcript'], return_tensors='np').input_ids[0]
            return {
                'input_features': torch.tensor(features, dtype=torch.float32),
                'labels':         torch.tensor(token_ids, dtype=torch.long),
                'language':       item['language']
            }

    lang_ds      = SingleLangDataset(lang_test)
    lang_results = phase2_whisper_trainer.evaluate(lang_ds)
    wer_val      = lang_results.get('eval_wer', 1.0) * 100
    lang_wers[lang_code] = wer_val
    lang_name    = lang_cfg.replace('studio-', '')
    status       = 'PASS' if wer_val <= WER_THRESHOLD else '  FAIL'
    print(f'  {lang_name:<12} {wer_val:>7.2f}%  {status}')

if lang_wers:
    vals = list(lang_wers.values())
    gap  = max(vals) - min(vals)
    print(f'\n  Max WER gap: {gap:.2f}pp | Threshold: 15pp')
    print('  FAIRNESS  :', 'Within threshold' if gap <= 15 else ' Exceeds threshold')

In [ ]:
# CELL B-12 │ Inference Demo - one sample per language
whisper_model.eval()
print('Whisper Inference Demo')
print('═' * 60)

seen_langs = set()
for sample in w_test.samples:
    lang_code = sample['language']
    if lang_code in seen_langs:
        continue
    seen_langs.add(lang_code)

    features = processor.feature_extractor(
        sample['waveform'], sampling_rate=WHISPER_SAMPLE_RATE, return_tensors='pt'
    ).input_features.to(DEVICE)

    processor.tokenizer.set_prefix_tokens(language=lang_code, task='transcribe')
    forced_ids = processor.get_decoder_prompt_ids(language=lang_code, task='transcribe')

    with torch.no_grad():
        pred_ids = whisper_model.generate(
            features, forced_decoder_ids=forced_ids, max_new_tokens=128
        )

    transcription = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]
    print(f'\n  [{lang_code.upper()}]')
    print(f'  Reference    : {sample["transcript"]}')
    print(f'  Transcription: {transcription}')

    if len(seen_langs) == NUM_LABELS:
        break

In [ ]:
# CELL B-13 │ Save Whisper Model
save_path_whisper = WHISPER_OUTPUT_DIR + '/final_model'
whisper_model.save_pretrained(save_path_whisper)
processor.save_pretrained(save_path_whisper)
print(f'Whisper saved → {save_path_whisper}')
for f in os.listdir(save_path_whisper):
    print(f'   {f}')